# 05b Protocol D deep: M0 / M1p vs residual comparators
Phase 5b only. Full Protocol D (3 repeats x 6 outer folds), nested tuning, scoring, paired statistics, pre-registered gate verdict. Smoke run first, then the full run, both from the last cell.

## 1. Setup and guards

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import sys, time, json, gc, shutil, hashlib, inspect, itertools, warnings, ctypes, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd, psutil
T_START = time.time()
RUN_M1 = False
SEED = 42

def keep_awake(on):
    try:
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000001 if on else 0x80000000)
    except Exception as e:
        print("keep-awake unavailable:", e)
keep_awake(True)

ROOT = Path.cwd(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
PROC = ROOT/"data/processed"; SPL = PROC/"splits"; P3 = ROOT/"outputs/phase3"; OUT4 = ROOT/"outputs/phase4"
OUT = ROOT/"outputs/phase5b"; FIG = OUT/"figures"; CACHE = OUT/"cache"; SRC = ROOT/"src"
for _p in (OUT, FIG, CACHE):
    _p.mkdir(parents=True, exist_ok=True)

CHECKS, ERRORS, EVENTS = [], [], []
def chk(level, name, detail=""):
    CHECKS.append((level, name, str(detail))); print(f"[{level}] {name} {detail}")
    if level == "FAIL":
        ERRORS.append(f"{name}: {detail}")
        raise AssertionError(f"{name}: {detail}")
def need(cond, name, detail="", level="FAIL"):
    chk("PASS" if cond else level, name, "" if cond else detail)

def sha256_file(p, crlf=False):
    b = Path(p).read_bytes()
    if crlf:
        b = b.replace(b"\r\n", b"\n")
    return hashlib.sha256(b).hexdigest()

def require_cols(df, cols, name):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise KeyError(f"{name}: missing column(s) {miss}; found {list(df.columns)}")

def ram_report(tag=""):
    av = psutil.virtual_memory().available / 1e9
    procs = []
    for p in psutil.process_iter(["name", "memory_info"]):
        try:
            procs.append((p.info["memory_info"].rss / 1e9, p.info["name"]))
        except Exception:
            pass
    procs.sort(reverse=True)
    print(f"RAM available {av:.1f} GB {tag}; top 8:", ", ".join(f"{n} {g:.1f}GB" for g, n in procs[:8]))
    if av < 2.0:
        chk("WARN", "RAM below 2 GB", f"{av:.1f} GB available; close memory hogs")
    return av
ram_report("(start)")

# gate / module guards
GATE_PATH = OUT4/"week2_gate.json"
GATE_SHA = "40df2fb82096e994f8bf7b8084433bb5ca148d3fa92c780461f68c0c4247d901"
MODULE_SHA_EXPECTED = "fc53e37d506ebe51f6e4d71819c4b047de5bc38456669a160183531fbefebea6"
def gate_check(tag):
    h = sha256_file(GATE_PATH)
    txt = (OUT4/"gate_hash.txt").read_text().split()[0].strip()
    need(h == txt == GATE_SHA, f"gate hash {tag}", f"file={h} gate_hash.txt={txt} expected={GATE_SHA}")
gate_check("start")
GATE = json.loads(GATE_PATH.read_text())
TARGETS = GATE["targets"]

sys.path.insert(0, str(SRC))
MODULE_SHA = sha256_file(SRC/"soh_deep_v2.py", crlf=True)
need(MODULE_SHA == MODULE_SHA_EXPECTED, "soh_deep_v2 hash", MODULE_SHA)
import soh_common as sc
import soh_deep_v2 as sd
import re
_PUB = [n for n in dir(sd) if not n.startswith("_")]
print("public names in soh_deep_v2:", _PUB)
print("KINDS:", getattr(sd, "KINDS", None))
def _sig(o):
    try: return str(inspect.signature(o))
    except Exception as e: return f"(signature unavailable: {e})"
_REQ = ["train_members", "predict", "fit_scalers", "sub_idx", "count_params"]
for _n in _REQ:
    if not hasattr(sd, _n):
        chk("FAIL", f"soh_deep_v2.{_n} missing", "this notebook uses the real module API: train_members, fit_scalers, predict, sub_idx, count_params")
for _n in _REQ + ["make_inputs", "base_pp", "cell_macro_mae", "lr_at", "epoch_positions", "clip_per_member", "make_batched"]:
    _o = getattr(sd, _n, None)
    if _o is not None:
        print(f"--- {_n}{_sig(_o)}\n{inspect.getdoc(_o)}\n")
_SRC = inspect.getsource(sd)
CFG_KEYS = sorted((set(re.findall(r'cfg\[\s*["\'](\w+)["\']\s*\]', _SRC)) | set(re.findall(r'cfg\.get\(\s*["\'](\w+)["\']', _SRC))
                   | set(re.findall(r'\bcfg\.([A-Za-z_]\w*)', _SRC))) - {"get", "items", "keys", "values", "update", "copy"})
print("cfg keys used by the module:", CFG_KEYS)
print("source lines mentioning warm/clip/noise/huber:")
for _ln in _SRC.splitlines():
    if any(t in _ln.lower() for t in ("warm", "huber", "noise")) and "cfg" in _ln: print("   ", _ln.strip())

import torch
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
need(torch.cuda.is_available(), "CUDA available")
_pr = torch.cuda.get_device_properties(0)
GPU_INFO = f"{_pr.name} {_pr.total_memory/1e9:.1f} GB, torch {torch.__version__}"
print("GPU:", GPU_INFO, "| TF32 matmul", torch.backends.cuda.matmul.allow_tf32)
sc.assert_allowed(["q0"], roles=("primary",))
chk("PASS", "allow-list q0 (primary)")

RAM available 5.4 GB (start); top 8: Code.exe 0.8GB, Code.exe 0.6GB, Code.exe 0.5GB, explorer.exe 0.4GB, MemCompression 0.4GB, claude.exe 0.4GB, MsMpEng.exe 0.3GB, chrome.exe 0.3GB
[PASS] gate hash start 
[PASS] soh_deep_v2 hash 
public names in soh_deep_v2: ['BatchedM0', 'BatchedM1', 'F', 'KINDS', 'MODULE_HASH_OK', 'Path', 'SingleM0', 'SingleM1', 'base_pp', 'cell_macro_mae', 'clip_per_member', 'count_params', 'epoch_positions', 'fit_scalers', 'hashlib', 'lr_at', 'make_batched', 'make_inputs', 'make_single', 'math', 'nn', 'np', 'parity_check', 'predict', 'same_seed_identical', 'self_hash', 'sub_idx', 'time', 'torch', 'train_members']
KINDS: ('M0', 'M1', 'M1p')
--- train_members(X, q0, y, members, kind, head, lr, epochs, cfg, device, dtype=torch.float32, log_every=10)
Train G members at once. members: dicts with seed, train_rows, train_w,
val_rows, val_codes. X (N,L), q0 (N,), y (N,) are tensors on `device`.

--- predict(model, X, q0, rows_list, sc, head, nominal, device, dtype, chunk=1

## 2. Data, delta-Q per window, outer sets

In [2]:
SENS_CELLS = {"b1c8", "b1c10", "b1c12", "b1c13", "b1c22"}
STITCHED = {"b1c0", "b1c1", "b1c2", "b1c3", "b1c4"}
WINDOWS = {"W310": 3.10, "W305": 3.05, "W300": 3.00, "W280": 2.80, "W240": 2.40, "WFULL": 2.00}
MINUTES = {"W310": 5.9, "W305": 8.6, "W300": 10.2, "W280": 12.4, "W240": 13.5, "WFULL": 13.9}
NARROW = {"W310", "W305", "W300"}          # G=72 ; wider windows G=36
DEEP = ("M0", "M1p", "M1"); COMP = ("B3r", "B4r")

_lp = [p for p in (PROC/"labels_table_v2.parquet", SPL/"labels_table_v2.parquet", P3/"labels_table_v2.parquet") if p.exists()]
need(bool(_lp), "labels_table_v2.parquet found", "looked in processed/, splits/, phase3/")
labels = pd.read_parquet(_lp[0])
require_cols(labels, ["row_id", "cell_id", "batch", "cycle_number", "q0", "soh_nom", "scored2"], "labels_table_v2")
need(len(labels) == 104971, "labels rows", len(labels))
need(bool((labels["row_id"].values == np.arange(len(labels))).all()), "row_id == position")
cycles, qdlin, t_knots, registry = sc.load_tables(root=ROOT, mmap=True)
need(qdlin.shape == (len(labels), 1000), "qdlin shape", qdlin.shape)
qref = np.load(PROC/"qref_f32.npy")
qref_ids = pd.read_csv(PROC/"qref_cell_ids.csv")
require_cols(qref_ids, ["row", "cell_id"], "qref_cell_ids.csv")
need(qref.shape == (129, 1000) and bool((qref_ids["row"].values == np.arange(len(qref_ids))).all()), "qref shape/rows", qref.shape)
QREF_ROW = dict(zip(qref_ids["cell_id"], qref_ids["row"]))

pD = pd.read_csv(SPL/"protocol_D.csv"); pI = pd.read_csv(SPL/"protocol_D_inner.csv")
require_cols(pD, ["repeat", "outer_fold", "cell_id", "role", "seed", "tertile"], "protocol_D.csv")
require_cols(pI, ["repeat", "outer_fold", "inner_fold", "cell_id", "role"], "protocol_D_inner.csv")
need(set(pD["role"]) <= {"train", "test"} and set(pI["role"]) <= {"train", "val"}, "split roles")
DEV_CELLS = sorted(pD["cell_id"].unique())
need(len(DEV_CELLS) == 84, "development cells", len(DEV_CELLS))
need(sum(c.startswith("b1") for c in DEV_CELLS) == 41 and sum(c.startswith("b2") for c in DEV_CELLS) == 43, "41 b1 + 43 b2")
need(not any(c.startswith("b3") for c in DEV_CELLS) and not (set(DEV_CELLS) & SENS_CELLS), "no batch-3 / sensitivity cells in development")
_t = pD[pD["role"] == "test"]
need(bool((_t.groupby(["repeat", "cell_id"]).size() == 1).all()) and bool((_t.groupby("repeat")["cell_id"].nunique() == 84).all()),
     "each dev cell in exactly one test set per repeat")
_n = _t.groupby(["repeat", "outer_fold"]).size()
need(len(_n) == 18 and _n.min() >= 13 and _n.max() <= 17, "18 outer sets with 13-17 test cells", _n.tolist())

_life = pd.read_csv(P3/"kept_cells_groups_strata.csv")
require_cols(_life, ["cell_id", "life"], "kept_cells_groups_strata.csv")
LIFE = dict(zip(_life["cell_id"], _life["life"]))

D = labels[labels["scored2"].astype(bool) & labels["cell_id"].isin(DEV_CELLS)]
D = D.sort_values(["cell_id", "cycle_number"], kind="stable").reset_index(drop=True)
need(len(D) == 55963, "development rows", len(D))
D["y"] = 100.0 * D["soh_nom"].astype(float)
D["life"] = D["cell_id"].map(LIFE).astype(float)
D["batch"] = D["batch"].astype(str)
need(not D["life"].isna().any(), "life known for every dev cell")
ROWID = D["row_id"].values.astype(np.int64); CELLID = D["cell_id"].values
Y = D["y"].values.astype(np.float64); Q0 = D["q0"].values.astype(np.float64)
CELL_POS = {c: np.asarray(ix) for c, ix in D.groupby("cell_id").indices.items()}
CELL_CODE = pd.factorize(D["cell_id"])[0]
def eval_pos(cells):
    return np.concatenate([CELL_POS[c] for c in sorted(cells)])
def cell_macro(pos, pred):
    ae = np.abs(np.asarray(pred, dtype=np.float64) - Y[pos]); cc = CELL_CODE[pos]
    s = np.bincount(cc, weights=ae); n = np.bincount(cc); return float((s[n > 0] / n[n > 0]).mean())

# frozen baselines
BASE = pd.read_csv(OUT4/"baseline_by_cell.csv")
print("baseline_by_cell.csv columns:", list(BASE.columns))
require_cols(BASE, ["model", "window", "cell_id", "mae_pp"], "baseline_by_cell.csv")
BASE = BASE.groupby(["model", "window", "cell_id"], as_index=False)["mae_pp"].mean()
need(bool((BASE.groupby(["model", "window"])["cell_id"].nunique() == 84).all()), "84 cells per (model, window) in baselines")

# delta-Q per window (explicit build_delta_q per cell block), cached as float32
CUR = {}
_DQKEY = hashlib.sha1(ROWID.tobytes()).hexdigest()[:10]
def get_dq(win):
    i_open = 0
    i_close = int(sc.grid_index(WINDOWS[win])); width = i_close + 1
    need(int(sc.grid_index(3.5)) == 0, "grid_index(3.5)==0") if win == "W310" else None
    fd, fw = CACHE/f"dq_{win}_{_DQKEY}.npy", CACHE/f"w_{win}_{_DQKEY}.npy"
    if fd.exists() and fw.exists():
        dq, w = np.load(fd), np.load(fw)
    else:
        dq = np.zeros((len(D), width), np.float32); w = np.zeros(len(D), np.float32)
        for c in DEV_CELLS:
            pos = CELL_POS[c]; blk = np.asarray(qdlin[ROWID[pos]], dtype=np.float32)
            out = np.asarray(sc.build_delta_q(blk, qref[QREF_ROW[c]], i_open, i_close))
            need(out.shape == (len(pos), width), f"build_delta_q shape {win} {c}", out.shape)
            dq[pos] = out; w[pos] = blk[:, i_close] - blk[:, i_open]
        np.save(fd, dq); np.save(fw, w)
    rng = np.random.default_rng(SEED)
    for r in rng.choice(len(D), 3, replace=False):
        one = np.asarray(sc.build_delta_q(np.asarray(qdlin[ROWID[r]], dtype=np.float32), qref[QREF_ROW[CELLID[r]]], i_open, i_close))
        need(np.abs(one.ravel() - dq[r]).max() < 1e-6 and abs(float(dq[r, 0])) < 1e-6 and dq.shape[1] == width,
             f"delta-Q check {win} row {r}")
    return dq, w, i_close

def use_window(win):
    CUR.clear(); gc.collect(); torch.cuda.empty_cache()
    dq, w, ic = get_dq(win)
    width = dq.shape[1]
    _g = sd.sub_idx(width); _g = _g.detach().cpu().numpy() if hasattr(_g, "detach") else np.asarray(_g)
    need(len(_g) == 32, "sub_idx gives 32 grid points", len(_g))
    CUR.update(win=win, dq=dq, w=w, i_close=ic, width=width, grid=_g.astype(int),
               Xt=torch.as_tensor(dq, device="cuda", dtype=torch.float32), q0t=torch.as_tensor(Q0, device="cuda", dtype=torch.float32),
               yt=torch.as_tensor(Y, device="cuda", dtype=torch.float32))
    ram_report(f"({win})")

def build_sets(cfg):
    keys = sorted(pD[["repeat", "outer_fold"]].drop_duplicates().itertuples(index=False, name=None))
    keys = cfg["outer_sets"] or keys
    forb = SENS_CELLS
    S = []
    for r, f in keys:
        g = pD[(pD["repeat"] == r) & (pD["outer_fold"] == f)]
        tr = sorted(g[g["role"] == "train"]["cell_id"]); te = sorted(g[g["role"] == "test"]["cell_id"])
        need(not set(tr) & set(te), f"train/test disjoint ({r},{f})")
        need(not any(c.startswith("b3") or c in forb for c in tr + te), f"no b3/sensitivity ({r},{f})")
        gi = pI[(pI["repeat"] == r) & (pI["outer_fold"] == f)]; inner = []
        for k in sorted(gi["inner_fold"].unique()):
            a = gi[gi["inner_fold"] == k]
            itr, iva = sorted(a[a["role"] == "train"]["cell_id"]), sorted(a[a["role"] == "val"]["cell_id"])
            need(not set(itr) & set(iva), f"inner train/val disjoint ({r},{f},{k})")
            need(set(itr) | set(iva) <= set(tr) and not (set(itr) | set(iva)) & set(te), f"tuning members exclude outer-test ({r},{f},{k})")
            inner.append(dict(inner_fold=int(k), train=itr, val=iva))
        need(len(inner) == 4, f"4 inner folds ({r},{f})", len(inner))
        S.append(dict(repeat=int(r), outer_fold=int(f), train=tr, test=te, inner=inner))
    return S

[PASS] labels_table_v2.parquet found 
[PASS] labels rows 
[PASS] row_id == position 
[PASS] qdlin shape 
[PASS] qref shape/rows 
[PASS] split roles 
[PASS] development cells 
[PASS] 41 b1 + 43 b2 
[PASS] no batch-3 / sensitivity cells in development 
[PASS] each dev cell in exactly one test set per repeat 
[PASS] 18 outer sets with 13-17 test cells 
[PASS] development rows 
[PASS] life known for every dev cell 
baseline_by_cell.csv columns: ['model', 'window', 'repeat', 'cell_id', 'batch', 'life', 'mae_pp']
[PASS] 84 cells per (model, window) in baselines 


## 3. Nested runner

In [3]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

# ---- config
def _grid_m0(first): g = [(lr, wd) for lr in (2e-3, 4e-3, 8e-3) for wd in (1e-4, 1e-3)]; return g[:1] if first else g
def _grid_m1p(first): g = [(lr, 1e-4) for lr in (4e-3, 8e-3)]; return g[:1] if first else g
FULL_CFG = dict(name="full", stage_a=["W300", "W240"], stage_b=["W310", "W305", "W280", "WFULL"], run_stage_b=True,
                outer_sets=None, E={"M0": 120, "M1p": 160, "M1": 160}, tune_seed=42, final_seeds=[42, 43, 44, 45, 46],
                first_only=False, trees_tune=150, trees_final=300, boot=5000, seed=SEED, full=True,
                out=OUT, fig=FIG, cache=CACHE)
_so = OUT/"smoke"
SMOKE_CFG = dict(name="smoke", stage_a=["W300"], stage_b=[], run_stage_b=False,
                 outer_sets=[(0, 0)], E={"M0": 2, "M1p": 2, "M1": 2}, tune_seed=42, final_seeds=[42, 43],
                 first_only=True, trees_tune=20, trees_final=20, boot=200, seed=SEED, full=False,
                 out=_so, fig=_so/"figures", cache=CACHE/"smoke")

def grid(cfg, model):
    f = cfg["first_only"]; H = []
    if model == "M0":
        for lr, wd in _grid_m0(f): H.append(dict(name=f"lr{lr:g}_wd{wd:g}", lr=lr, wd=wd, ord=(lr, wd)))
    elif model in ("M1p", "M1"):
        for lr, wd in _grid_m1p(f): H.append(dict(name=f"lr{lr:g}_wd{wd:g}", lr=lr, wd=wd, ord=(lr, wd)))
    elif model == "B3r":
        for d in (2, 3, 4):
            for lr in (0.05, 0.1): H.append(dict(name=f"d{d}_lr{lr:g}", depth=d, lr=lr, wd=np.nan, ord=(d, lr)))
        H = H[:1] if f else H
    elif model == "B4r":
        for leaf in (1, 2, 3, 5): H.append(dict(name=f"leaf{leaf}", leaf=leaf, lr=np.nan, wd=np.nan, ord=(leaf,)))
        H = H[:1] if f else H
    return H

def stage_plan(cfg):
    deep = ["M0", "M1p"] + (["M1"] if RUN_M1 else [])
    P = [("A", cfg["stage_a"], deep + list(COMP))]
    if cfg["run_stage_b"] and cfg["stage_b"]:
        P.append(("B", cfg["stage_b"], ["M0"] + list(COMP)))
    return P

# ---- logging
RUNS, WORK, TUNE_LOG, SEL_ROWS = [], {"deep": [0.0, 0.0], "comp": [0.0, 0.0]}, [], []
LOGFILE = [OUT/"run_log.txt"]
def logged(scope, name, kind, fn, units=0.0, wk=None):
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    t0 = time.time(); res = fn(); secs = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    if wk: WORK[wk][0] += secs; WORK[wk][1] += units
    prev = [r["seconds"] for r in RUNS if r["kind"] == kind and r["status"] == "run"]
    if prev and secs > 3 * float(np.median(prev)):
        chk("WARN", "slow run", f"{name}: {secs:.0f}s > 3x median {np.median(prev):.0f}s")
    RUNS.append(dict(ts=dt.datetime.now().isoformat(timespec="seconds"), scope=scope, name=name, kind=kind, seconds=secs, status="run", peak_gb=peak))
    with open(LOGFILE[0], "a") as fh: fh.write(f"{RUNS[-1]['ts']}\t{scope}\t{name}\t{secs:.1f}\trun\t{peak:.2f}\n")
    return res
def log_cached(scope, name, kind):
    RUNS.append(dict(ts=dt.datetime.now().isoformat(timespec="seconds"), scope=scope, name=name, kind=kind, seconds=0.0, status="cached", peak_gb=0.0))
    with open(LOGFILE[0], "a") as fh: fh.write(f"{RUNS[-1]['ts']}\t{scope}\t{name}\t0.0\tcached\t0.00\n")

def ckey(cfg, stage, win, model, hp, seeds, E, sets):
    d = dict(module=MODULE_SHA, cfg=cfg["name"], stage=stage, window=win, model=model, hp={k: (None if isinstance(v, float) and np.isnan(v) else v) for k, v in hp.items()},
             seeds=list(seeds), E=E, sets=[(s["repeat"], s["outer_fold"]) for s in sets], trees=[cfg["trees_tune"], cfg["trees_final"]])
    return hashlib.sha1(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:20]

# ---- adapter to the real soh_deep_v2 API: train_members / fit_scalers / predict / count_params
def _np(x):
    return x.detach().cpu().numpy() if hasattr(x, "detach") else np.asarray(x)

class Cfg(dict):
    def __getattr__(self, k):
        try: return self[k]
        except KeyError: raise AttributeError(k)

def make_cfg(hp, E):
    v = dict(wd=hp["wd"], weight_decay=hp["wd"], lr=hp["lr"], batch=256, batch_size=256, bs=256, clip=1.0, grad_clip=1.0, noise=0.02, noise_sd=0.02,
             sigma=0.02, delta=1.0, huber=1.0, huber_delta=1.0, warm=0.05, warmup=0.05, warm_frac=0.05, warmup_frac=0.05, nominal=1.1, epochs=E, E=E, head="abs",
             BATCH=256, CLIP=1.0, HUBER_DELTA=1.0, NOISE_SIGMA=0.02, NOMINAL_AH=1.1, WARMUP_FRAC=0.05, WD=hp["wd"])
    miss = [k for k in CFG_KEYS if k not in v]
    if miss: raise RuntimeError(f"cfg key(s) {miss} are used by soh_deep_v2 but have no value in make_cfg; add them (see the printed source lines in section 1)")
    return Cfg(v)

def train_rows(cells, seed):
    # every 5th scored2 row per cell with a random per-cell offset; weights 1/(rows of that cell)
    rng = np.random.default_rng(seed); pos, w = [], []
    for c in sorted(cells):
        p = CELL_POS[c]; q = p[int(rng.integers(0, 5))::5]
        pos.append(q); w.append(np.full(len(q), 1.0 / len(q)))
    pos, w = np.concatenate(pos).astype(np.int64), np.concatenate(w).astype(np.float32)
    need(len(pos) > 0 and set(CELLID[pos]) <= set(cells), "train rows valid and inside training cells")
    return pos, w

def _data(mode):
    if mode == "n": return CUR["dq"], Q0.astype(np.float32), Y.astype(np.float32)
    return CUR["Xt"], CUR["q0t"], CUR["yt"]
def _rows(mode, arrs):
    return [torch.as_tensor(a, device="cuda") for a in arrs] if mode == "tt" else list(arrs)
def _try_modes(f):
    first = None
    for mode in ("t", "tt", "n"):          # GPU tensors, GPU tensors incl. row lists, then numpy
        try: return f(mode)
        except torch.cuda.OutOfMemoryError: raise
        except Exception as e: first = first or e
    raise first
def _pick_model(ret):
    cands = list(ret.values()) if isinstance(ret, dict) else list(ret) if isinstance(ret, (tuple, list)) else [ret]
    for c in cands:
        if hasattr(c, "parameters"): return c
    raise RuntimeError(f"train_members returned {type(ret)}; no torch module found in it")

def split_preds(out, lens):
    n = len(lens)
    if isinstance(out, (list, tuple)): arrs = [_np(a).reshape(-1) for a in out]
    else:
        a = _np(out)
        if a.ndim == 2 and a.shape[0] == n: arrs = [a[i] for i in range(n)]
        elif a.ndim == 1 and a.size == sum(lens): arrs = np.split(a, np.cumsum(lens)[:-1])
        else: raise RuntimeError(f"predict output shape {a.shape} does not match {n} members / lengths {lens[:3]}...")
    arrs = [a[:L] for a, L in zip(arrs, lens)]          # tolerate padding
    for a, L in zip(arrs, lens): need(len(a) == L, "prediction length matches eval rows", f"{len(a)} vs {L}")
    return [a.astype(np.float64) for a in arrs]

_PARAM_CHECKED = set()
def deep_chunk(win, model, hp, E, chunk):
    # train_members trains the whole chunk at once and returns val_pred for each member's val_rows (it returns no model).
    # tuning members: val_rows = inner-val rows; final members: val_rows = outer-test rows (predicted once at the final epoch,
    # no early stopping, nothing selected from them).
    tr = [train_rows(m["train"], m["seed"]) for m in chunk]; ev = [eval_pos(m["eval"]) for m in chunk]
    cfg = make_cfg(hp, E)
    mem = [dict(seed=m["seed"], train_rows=r, train_w=w, val_rows=e, val_codes=pd.factorize(CELLID[e])[0].astype(np.int64))
           for m, (r, w), e in zip(chunk, tr, ev)]
    ret = sd.train_members(X=CUR["Xt"], q0=CUR["q0t"], y=CUR["yt"], members=mem, kind=model, head="abs", lr=hp["lr"], epochs=E, cfg=cfg,
                           device=torch.device("cuda"), dtype=torch.float32, log_every=10**6)
    need(isinstance(ret, dict) and "val_pred" in ret, "train_members returned val_pred", type(ret))
    preds = split_preds(ret["val_pred"], [len(e) for e in ev])
    nan_m = list(ret.get("nan_members", []))
    if nan_m: chk("WARN", "NaN members", f"{win}/{model}/{hp['name']}: members {nan_m[:10]}")
    if model not in _PARAM_CHECKED:
        _PARAM_CHECKED.add(model)
        try:
            npar = int(sd.count_params(sd.make_single(kind=model, L=CUR["width"], seed=0))); exp = {"M0": 2969, "M1p": 3049, "M1": 2977}[model]
            need(npar == exp, f"count_params {model}", f"{npar} vs {exp}", level="WARN")
        except Exception as e:
            chk("WARN", f"count_params {model} not checked", repr(e)[:200])
    for a in preds:
        if np.isfinite(a).all() and not (0 < a.mean() < 150):
            chk("WARN", "prediction range suspicious (expected pp)", f"mean {a.mean():.2f}")
    del ret, mem; gc.collect(); torch.cuda.empty_cache()
    return preds

def run_chunked(fn, items, G, label):
    out, i = [], 0
    while i < len(items):
        ch = items[i:i + G]
        try:
            out += fn(ch); i += len(ch)
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache()
            if G == 1: raise
            G = max(1, G // 2); EVENTS.append(f"CUDA OOM in {label}: G halved to {G}")
            chk("WARN", "CUDA OOM", f"{label}: retrying with G={G}")
    return out

def gcap(win): return 72 if win in NARROW else 36

def pred_frame(win, model, rep, of, seed, ep, pred):
    return pd.DataFrame(dict(window=win, model=model, repeat=rep, outer_fold=of, seed=seed, row_id=ROWID[ep], cell_id=CELLID[ep], pred_pp=pred))

# ---- tuning
def tune(cfg, win, model, hp, sets):
    E = cfg["E"].get(model, 0); seeds = [cfg["tune_seed"]]
    fp = cfg["cache"]/f"tune_{ckey(cfg, 'tune', win, model, hp, seeds, E, sets)}.parquet"; nm = f"{win}/{model}/{hp['name']}"
    if fp.exists():
        log_cached("tune", nm, f"tune:{model}"); return pd.read_parquet(fp)
    rows = []
    if model in DEEP:
        mem = [dict(train=i["train"], eval=i["val"], seed=cfg["tune_seed"], stage="tune", tag=(s["repeat"], s["outer_fold"], i["inner_fold"])) for s in sets for i in s["inner"]]
        preds = logged("tune", nm, f"tune:{model}", lambda: run_chunked(lambda ch: deep_chunk(win, model, hp, E, ch), mem, gcap(win), nm),
                       units=len(mem) * E, wk="deep")
        for m, p in zip(mem, preds):
            rows.append(dict(window=win, model=model, config=hp["name"], repeat=m["tag"][0], outer_fold=m["tag"][1], inner_fold=m["tag"][2],
                             mae=float(np.nan_to_num(cell_macro(eval_pos(m["eval"]), p), nan=np.inf))))
    else:
        def go():
            res = []
            for s in sets:
                for i in s["inner"]:
                    ep, p = comp_fit_predict(model, hp, i["train"], i["val"], cfg["tune_seed"], cfg["trees_tune"])
                    res.append(dict(window=win, model=model, config=hp["name"], repeat=s["repeat"], outer_fold=s["outer_fold"], inner_fold=i["inner_fold"], mae=cell_macro(ep, p)))
            return res
        rows = logged("tune", nm, f"tune:{model}", go, units=len(sets) * 4, wk="comp")
    T = pd.DataFrame(rows); T.to_parquet(fp); return T

def select(T, hps):
    by = {h["name"]: h for h in hps}; sel = {}
    m = T.groupby(["repeat", "outer_fold", "config"])["mae"].mean().reset_index()
    for (r, f), g in m.groupby(["repeat", "outer_fold"]):
        b = min(g.itertuples(), key=lambda t: (round(t.mae, 10),) + tuple(by[t.config]["ord"]))
        sel[(int(r), int(f))] = dict(by[b.config], inner_mean_mae=float(b.mae))
    return sel

# ---- comparators (CPU)
def comp_X(model):
    return np.column_stack([CUR["w"], Q0]) if model == "B3r" else np.column_stack([CUR["dq"][:, CUR["grid"]], Q0])
def comp_fit_predict(model, hp, train_cells, eval_cells, seed, trees):
    pos, w = train_rows(train_cells, seed); X = CUR.setdefault("X_" + model, comp_X(model))
    off = 100.0 * Q0 / 1.1; r = Y - off
    if model == "B3r":
        est = HistGradientBoostingRegressor(max_depth=hp["depth"], learning_rate=hp["lr"], max_iter=200, early_stopping=False, random_state=seed)
    else:
        est = RandomForestRegressor(n_estimators=trees, min_samples_leaf=hp["leaf"], max_features=0.33, n_jobs=10, random_state=seed)
    est.fit(X[pos], r[pos], sample_weight=w)
    ep = eval_pos(eval_cells)
    return ep, est.predict(X[ep]) + off[ep]

# ---- final
def final(cfg, win, model, hps, sel, sets):
    E = cfg["E"].get(model, 0); seeds = cfg["final_seeds"] if model in DEEP else cfg["final_seeds"][:1]
    groups = {}
    for s in sets: groups.setdefault(sel[(s["repeat"], s["outer_fold"])]["name"], []).append(s)
    frames = []
    for name, ss in groups.items():
        hp = next(h for h in hps if h["name"] == name); nm = f"{win}/{model}/{name}"
        fp = cfg["cache"]/f"final_{ckey(cfg, 'final', win, model, hp, seeds, E, ss)}.parquet"
        if fp.exists():
            log_cached("final", nm, f"final:{model}"); frames.append(pd.read_parquet(fp)); continue
        if model in DEEP:
            mem = [dict(train=s["train"], eval=s["test"], seed=sd_, stage="final", tag=(s["repeat"], s["outer_fold"])) for s in ss for sd_ in seeds]
            preds = logged("final", nm, f"final:{model}", lambda: run_chunked(lambda ch: deep_chunk(win, model, hp, E, ch), mem, gcap(win), nm),
                           units=len(mem) * E, wk="deep")
            fr = pd.concat([pred_frame(win, model, m["tag"][0], m["tag"][1], m["seed"], eval_pos(m["eval"]), p) for m, p in zip(mem, preds)], ignore_index=True)
        else:
            def go():
                out = []
                for s in ss:
                    ep, p = comp_fit_predict(model, hp, s["train"], s["test"], seeds[0], cfg["trees_final"])
                    out.append(pred_frame(win, model, s["repeat"], s["outer_fold"], seeds[0], ep, p))
                return pd.concat(out, ignore_index=True)
            fr = logged("final", nm, f"final:{model}", go, units=len(ss), wk="comp")
        fr.to_parquet(fp); frames.append(fr)
    return pd.concat(frames, ignore_index=True)

def run_model(cfg, win, model, sets):
    hps = grid(cfg, model)
    T = pd.concat([tune(cfg, win, model, hp, sets) for hp in hps], ignore_index=True)
    TUNE_LOG.append(T); sel = select(T, hps)
    for (r, f), h in sel.items():
        SEL_ROWS.append(dict(window=win, model=model, repeat=r, outer_fold=f, config=h["name"], lr=h.get("lr"), wd=h.get("wd"), inner_mean_mae=h["inner_mean_mae"]))
    return final(cfg, win, model, hps, sel, sets)

def print_estimates(cfg, sets):
    rd = WORK["deep"][0] / WORK["deep"][1] if WORK["deep"][1] else None
    rc = WORK["comp"][0] / WORK["comp"][1] if WORK["comp"][1] else None
    print("Estimated work (rate from smoke when available; rough):")
    for st, wins, models in stage_plan(cfg):
        tot = 0.0; desc = []
        for win in wins:
            for mo in models:
                nh = len(grid(cfg, mo))
                if mo in DEEP:
                    units = (nh * len(sets) * 4 + len(sets) * len(cfg["final_seeds"])) * cfg["E"][mo]
                    tot += units * rd if rd else 0
                else:
                    units = nh * len(sets) * 4 + len(sets)
                    tot += units * rc * (cfg["trees_tune"] / 20 if mo == "B4r" and cfg["name"] == "full" else 1) if rc else 0
                desc.append(f"{win}/{mo}:{units:.0f}")
        print(f"  Stage {st}: {len(wins)} windows, work units {', '.join(desc[:6])}{'...' if len(desc) > 6 else ''}  ~{tot/60:.0f} min" if (rd or rc) else f"  Stage {st}: units {', '.join(desc)}")

## 4. Scoring

In [4]:
def _stage_label(lf):
    return np.minimum((lf * 10).astype(int), 9)

def score_all(cfg, P):
    info = D[["row_id", "y", "cycle_number", "life", "batch"]]
    P = P.merge(info, on="row_id", how="left"); P["ae1"] = (P["pred_pp"] - P["y"]).abs()
    E_ = P.groupby(["window", "model", "repeat", "row_id"], sort=False).agg(
        pred=("pred_pp", "mean"), y=("y", "first"), cell_id=("cell_id", "first"), cycle_number=("cycle_number", "first"),
        life=("life", "first"), batch=("batch", "first")).reset_index()
    E_["err"] = E_["pred"] - E_["y"]; E_["ae"] = E_["err"].abs(); E_["lf"] = E_["cycle_number"] / E_["life"]; E_["tenth"] = _stage_label(E_["lf"].values)
    K = ["window", "model"]
    cr = E_.groupby(K + ["repeat", "cell_id"]).agg(mae=("ae", "mean"), life=("life", "first"), batch=("batch", "first")).reset_index()
    sc_ = P.groupby(K + ["repeat", "seed", "cell_id"])["ae1"].mean().reset_index()
    by_cell = cr.groupby(K + ["cell_id"]).agg(mae=("mae", "mean"), life=("life", "first"), batch=("batch", "first")).reset_index()
    single_cell = sc_.groupby(K + ["cell_id"])["ae1"].mean().reset_index().rename(columns={"ae1": "mae"})
    res = []
    for (w, m), g in cr.groupby(K):
        per = g.groupby("repeat")["mae"].mean(); e = E_[(E_["window"] == w) & (E_["model"] == m)]
        s1 = sc_[(sc_["window"] == w) & (sc_["model"] == m)]; sm = s1.groupby(["repeat", "seed"])["ae1"].mean()
        bc = by_cell[(by_cell["window"] == w) & (by_cell["model"] == m)].sort_values("mae", ascending=False)
        ex = bc[~bc["cell_id"].isin(STITCHED)]
        sd_seed = sm.groupby("repeat").std(ddof=1).mean() if sm.groupby("repeat").size().min() > 1 else np.nan
        res.append(dict(window=w, model=m, macro_mean=per.mean(), macro_sd=per.std(ddof=1) if len(per) > 1 else np.nan, n_repeats=len(per),
                        single_seed_macro=sm.mean(), seed_sd=sd_seed, pooled_mae=e["ae"].mean(),
                        rmspe_pct=100 * float(np.sqrt(np.mean((e["err"] / e["y"]) ** 2))), bias=e["err"].mean(),
                        cell_median=bc["mae"].median(), cell_p90=bc["mae"].quantile(0.9), macro_excl_stitched=ex["mae"].mean(),
                        median_excl_stitched=ex["mae"].median(), p90_excl_stitched=ex["mae"].quantile(0.9),
                        last20_macro=e[e["lf"] >= 0.8].groupby(["repeat", "cell_id"])["ae"].mean().mean(),
                        worst5="; ".join(f"{r.cell_id}(life {r.life:.0f}, {r.batch}, {r.mae:.2f})" for r in bc.head(5).itertuples())))
    res = pd.DataFrame(res)
    st = E_.groupby(K + ["repeat", "cell_id", "tenth"])["ae"].mean().reset_index().groupby(K + ["tenth"])["ae"].mean().reset_index().rename(columns={"ae": "mae", "tenth": "stage"})
    by_batch = by_cell.groupby(K + ["batch"])["mae"].agg(["mean", "median", "count"]).reset_index().rename(columns={"mean": "macro_mae"})
    cell_tenth = E_.groupby(K + ["cell_id", "tenth"])["ae"].mean().reset_index()
    o = cfg["out"]; o.mkdir(parents=True, exist_ok=True)
    res.to_csv(o/"deep_results.csv", index=False); by_cell.to_csv(o/"deep_by_cell.csv", index=False)
    st.to_csv(o/"deep_by_stage.csv", index=False); by_batch.to_csv(o/"deep_by_batch.csv", index=False)
    if cfg["full"]:
        for (w, m), g in E_.groupby(K):
            need(len(g) == 3 * len(D) and g["row_id"].nunique() == len(D), f"full coverage {w}/{m}", len(g))
    return dict(E=E_, by_cell=by_cell, single_cell=single_cell, res=res, stage=st, by_batch=by_batch, cell_tenth=cell_tenth)

## 5. Statistics: cluster bootstrap over cells

In [5]:
_IDX = {}
def get_idx(n, boot, seed):
    k = (n, boot, seed)
    if k not in _IDX: _IDX[k] = np.random.default_rng(seed).integers(0, n, (boot, n))
    return _IDX[k]

def frozen_series(model, win):
    g = BASE[(BASE["model"] == model) & (BASE["window"] == win)]
    return pd.Series(g["mae_pp"].values, index=g["cell_id"].values) if len(g) else None

def stats(cfg, S):
    rows, pairs = [], {}; boot, seed = cfg["boot"], cfg["seed"]
    cells = {(w, m): g.set_index("cell_id")["mae"] for (w, m), g in S["by_cell"].groupby(["window", "model"])}
    singles = {(w, m): g.set_index("cell_id")["mae"] for (w, m), g in S["single_cell"].groupby(["window", "model"])}
    def ci(w, m, a):
        idx = get_idx(len(a), boot, seed); b = a.values[idx].mean(1)
        rows.append(dict(window=w, kind="ci", a=m, b="", point=a.mean(), lo=np.percentile(b, 2.5), hi=np.percentile(b, 97.5), ratio=np.nan, ratio_lo=np.nan, ratio_hi=np.nan, n_cells=len(a)))
    def pair(w, an, bn, a, b, tag=""):
        if a is None or b is None: return
        c = a.index.intersection(b.index)
        if len(c) == 0: return
        if cfg["full"]: need(len(c) == 84, f"84 paired cells {w} {an} vs {bn}", len(c))
        a, b = a.loc[c], b.loc[c]; idx = get_idx(len(c), boot, seed)
        d = (a.values - b.values)[idx].mean(1); r = a.values[idx].mean(1) / b.values[idx].mean(1)
        rec = dict(window=w, kind="paired" + tag, a=an, b=bn, point=float((a - b).mean()), lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5),
                   ratio=float(a.mean() / b.mean()), ratio_lo=np.percentile(r, 2.5), ratio_hi=np.percentile(r, 97.5), n_cells=len(c))
        rows.append(rec); pairs[(w, an, bn)] = rec
    for w in sorted(set(S["res"]["window"])):
        for m in sorted(set(S["res"][S["res"]["window"] == w]["model"])): ci(w, m, cells[(w, m)])
        bb = TARGETS.get(w, {}).get("best_baseline"); fb = frozen_series(bb, w) if bb else None
        m0 = cells.get((w, "M0")); m1 = cells.get((w, "M1p"))
        comps = [c for c in COMP if (w, c) in cells]
        bc = min(comps, key=lambda c: cells[(w, c)].mean()) if comps else None
        pair(w, "M0", f"frozen:{bb}", m0, fb)
        if bc: pair(w, "M0", bc, m0, cells[(w, bc)])
        if m1 is not None:
            pair(w, "M1p", "M0", m1, m0); pair(w, "M1p", f"frozen:{bb}", m1, fb)
            if bc: pair(w, "M1p", bc, m1, cells[(w, bc)])
        for m in ("M0", "M1p"):
            if (w, m) in singles: pair(w, f"{m} ensemble", f"{m} single-seed", cells[(w, m)], singles[(w, m)])
    B = pd.DataFrame(rows); cfg["out"].mkdir(parents=True, exist_ok=True); B.to_csv(cfg["out"]/"bootstrap_5b.csv", index=False)
    return dict(boot=B, pairs=pairs)

## 6. Pre-registered gate verdict (written once)

In [6]:
def write_verdict(cfg, S, ST):
    path = cfg["out"]/"gate_verdict.json"
    if path.exists(): raise FileExistsError(f"{path} exists; the verdict is written once and never overwritten")
    macro = S["res"].set_index(["window", "model"])["macro_mean"]
    V = dict(created=dt.datetime.now().isoformat(timespec="seconds"), config=cfg["name"], primary_model="M0", boot=cfg["boot"], seed=cfg["seed"],
             gate_sha256=GATE_SHA, module_sha256=MODULE_SHA, rule="PASS iff ensemble cell-macro MAE <= m1_target_pp AND upper 95% bound of paired diff (M0 - frozen best baseline) < 0",
             headline="W300", companion="W240", windows={})
    for w in sorted(set(S["res"]["window"])):
        t = TARGETS.get(w); e = dict(role="headline" if w == "W300" else "companion" if w == "W240" else "descriptive")
        if (w, "M0") in macro.index: e["m0_ensemble_cell_macro_mae_pp"] = float(macro[(w, "M0")])
        if t and (w, "M0") in macro.index:
            bb = t["best_baseline"]; p = ST["pairs"].get((w, "M0", f"frozen:{bb}"))
            e.update(target_m1_pp=float(t["m1_target_pp"]), best_baseline=bb, best_baseline_cell_macro_mae_pp=float(t["best_cell_macro_mae_pp"]))
            if p:
                e.update(paired_diff_vs_best_baseline=dict(point=p["point"], lo=p["lo"], hi=p["hi"]))
                e["pass_mae"] = bool(e["m0_ensemble_cell_macro_mae_pp"] <= e["target_m1_pp"]); e["pass_ci"] = bool(p["hi"] < 0)
                e["PASS"] = bool(e["pass_mae"] and e["pass_ci"])
        for tag, mod in (("m0_vs_better_comparator", "M0"), ("m1p_exploratory_vs_better_comparator", "M1p")):
            for (ww, a, b), p in ST["pairs"].items():
                if ww == w and a == mod and b in COMP:
                    e[tag] = dict(comparator=b, ratio=p["ratio"], ratio_ci=[p["ratio_lo"], p["ratio_hi"]], diff=p["point"], diff_ci=[p["lo"], p["hi"]], note="no pass/fail" + (" (exploratory)" if mod == "M1p" else ""))
        for (ww, a, b), p in ST["pairs"].items():
            if ww == w and a == "M1p" and b.startswith("frozen:"):
                e["m1p_exploratory_vs_frozen_best"] = dict(diff=p["point"], ci=[p["lo"], p["hi"]], note="exploratory")
        V["windows"][w] = e
    V["headline_PASS"] = V["windows"].get("W300", {}).get("PASS"); V["companion_PASS"] = V["windows"].get("W240", {}).get("PASS")
    path.write_text(json.dumps(V, indent=2))
    h = hashlib.sha256(path.read_bytes()).hexdigest(); (cfg["out"]/"gate_verdict_hash.txt").write_text(h + "\n")
    return V

## 7. Figures

In [7]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

def frozen_by_tenth():
    for f in OUT4.glob("*.csv"):
        try: h = pd.read_csv(f, nrows=3)
        except Exception: continue
        c = set(h.columns); tc = [x for x in ("tenth", "life_bin", "decile", "stage") if x in c]; mc = [x for x in ("mae_pp", "mae") if x in c]
        if {"model", "window"} <= c and tc and mc: return pd.read_csv(f), tc[0], mc[0]
    return None

def figures(cfg, S, ST):
    fd = cfg["fig"]; fd.mkdir(parents=True, exist_ok=True); res = S["res"]
    # 1 MAE vs minutes
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for m, mk in (("M0", "o"), ("M1p", "s"), ("B3r", "^"), ("B4r", "v")):
        g = res[res["model"] == m].sort_values("window", key=lambda s: s.map(MINUTES))
        if len(g): ax.plot(g["window"].map(MINUTES), g["macro_mean"], mk + "-", label=m)
    for m, mk in (("B1", "D"), ("B3", "^"), ("B4", "v")):
        g = BASE[BASE["model"] == m].groupby("window")["mae_pp"].mean(); g = g[g.index.isin(MINUTES)]
        if len(g): ax.scatter(g.index.map(MINUTES), g.values, marker=mk, facecolors="none", edgecolors="gray", label=f"frozen {m}")
    tg = [(MINUTES[w], t["m1_target_pp"]) for w, t in TARGETS.items() if w in MINUTES and "m1_target_pp" in t]
    if tg: ax.scatter(*zip(*tg), marker="*", s=90, facecolors="none", edgecolors="k", label="gate target")
    ax.set_xlabel("measured minutes at window close"); ax.set_ylabel("cell-macro MAE (pp)"); ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(fd/"mae_vs_minutes.png", dpi=150); plt.close(fig)
    # 2 error by tenth of life at W300
    fig, ax = plt.subplots(figsize=(7, 4.5)); st = S["stage"]
    for m in ("M0", "M1p", "B4r"):
        g = st[(st["window"] == "W300") & (st["model"] == m)]
        if len(g): ax.plot(g["stage"], g["mae"], "o-", label=m)
    fz = frozen_by_tenth()
    if fz is not None:
        df, tc, mc = fz; g = df[(df["window"] == "W300") & (df["model"] == "B4")].groupby(tc)[mc].mean()
        if len(g): ax.plot(np.arange(len(g)), g.values, "x--", label="B4 (frozen)")
    else:
        chk("WARN", "no frozen per-tenth baseline file found in phase4", "B4 line omitted from tenth-of-life figure")
    ax.set_xlabel("tenth of life"); ax.set_ylabel("MAE (pp)"); ax.legend(); fig.tight_layout(); fig.savefig(fd/"error_by_tenth_W300.png", dpi=150); plt.close(fig)
    # 3 heat map
    ct = S["cell_tenth"]; g = ct[(ct["window"] == "W300") & (ct["model"] == "M0")]
    if len(g):
        pv = g.pivot_table(index="cell_id", columns="tenth", values="ae"); order = sorted(pv.index, key=lambda c: LIFE[c]); pv = pv.loc[order]
        fig, ax = plt.subplots(figsize=(6, 12)); im = ax.imshow(pv.values, aspect="auto", cmap="viridis"); fig.colorbar(im, label="MAE (pp)")
        ax.set_yticks(range(len(order))); ax.set_yticklabels([f"{c} ({LIFE[c]:.0f})" for c in order], fontsize=5); ax.set_xlabel("tenth of life"); ax.set_title("M0 W300 per-cell error, sorted by life")
        fig.tight_layout(); fig.savefig(fd/"heatmap_M0_W300.png", dpi=150); plt.close(fig)
    # 4 forest plot
    B = ST["boot"]; B = B[B["kind"].str.startswith("paired")]
    if len(B):
        lab = [f"{r.window}: {r.a} - {r.b}" for r in B.itertuples()]; y = np.arange(len(B))[::-1]
        fig, ax = plt.subplots(figsize=(8, 0.3 * len(B) + 1.5)); ax.errorbar(B["point"], y, xerr=[B["point"] - B["lo"], B["hi"] - B["point"]], fmt="o", capsize=2)
        ax.axvline(0, color="k", lw=0.8); ax.set_yticks(y); ax.set_yticklabels(lab, fontsize=6); ax.set_xlabel("paired difference in cell MAE (pp); negative favours first")
        fig.tight_layout(); fig.savefig(fd/"forest_paired.png", dpi=150); plt.close(fig)

## 8. Report and pipeline

In [8]:
def _tbl(df, cols):
    return df[cols].to_string(index=False, float_format=lambda v: f"{v:.3f}")

def build_report(cfg, S, ST, V):
    L = ["PHASE 5B REPORT", f"config: {cfg['name']}  runtime: {(time.time() - T_START)/60:.1f} min  seed {cfg['seed']}  boot {cfg['boot']}",
         f"soh_deep_v2 sha256: {MODULE_SHA}", f"week2_gate sha256: {sha256_file(GATE_PATH)}", f"GPU: {GPU_INFO}",
         f"RAM available: {psutil.virtual_memory().available/1e9:.1f} GB", ""]
    res = S["res"]
    for w in sorted(set(res["window"]), key=lambda x: MINUTES[x]):
        L.append(f"== {w} ({MINUTES[w]} min) ==")
        L.append(_tbl(res[res["window"] == w], ["model", "macro_mean", "macro_sd", "single_seed_macro", "seed_sd", "pooled_mae", "rmspe_pct", "bias", "cell_median", "cell_p90", "macro_excl_stitched"]))
        fz = BASE[BASE["window"] == w].groupby("model")["mae_pp"].mean()
        L.append("frozen (cell-macro): " + ", ".join(f"{m} {v:.3f}" for m, v in fz.items() if m in ("B0b", "B1", "B2", "B3", "B4", "B5")))
        L.append("")
    L.append("== paired intervals (95% cluster bootstrap) ==")
    B = ST["boot"]; L.append(B[B["kind"] != "ci"][["window", "a", "b", "point", "lo", "hi", "ratio", "n_cells"]].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    L += ["", "== gate verdict ==", f"headline W300 PASS={V.get('headline_PASS')}  companion W240 PASS={V.get('companion_PASS')}"]
    for w, e in V["windows"].items():
        L.append(f"  {w} [{e['role']}]: M0 {e.get('m0_ensemble_cell_macro_mae_pp', float('nan')):.3f} target {e.get('target_m1_pp', float('nan')):.3f} "
                 f"pass_mae={e.get('pass_mae')} pass_ci={e.get('pass_ci')} PASS={e.get('PASS')}")
    sc_ = pd.DataFrame(SEL_ROWS)
    if len(sc_): L += ["", "== selected configs (count) ==", sc_.groupby(["window", "model", "config"]).size().to_string()]
    rr = pd.DataFrame(RUNS)
    if len(rr): L += ["", "== slowest runs ==", rr.sort_values("seconds", ascending=False).head(5)[["scope", "name", "seconds", "peak_gb"]].to_string(index=False)]
    L += ["", "== checks =="]
    for lv in ("PASS", "WARN", "FAIL"):
        items = [f"{n} {d}".strip() for l, n, d in CHECKS if l == lv]
        L.append(f"{lv}: {len(items)}"); L += ["   " + i for i in (items if lv != "PASS" else items[:0])]
    L += ["", "== errors ==", *(ERRORS or ["none"]), *EVENTS]
    arts = [(str(p.relative_to(cfg["out"])), p.stat().st_size) for p in sorted(cfg["out"].rglob("*")) if p.is_file() and p.suffix != ".parquet" and "cache" not in p.parts]
    L += ["", "== artifacts =="] + [f"  {n}  {s/1024:.1f} KB" for n, s in arts]
    txt = "\n".join(L)
    (cfg["out"]/"phase5b_report.txt").write_text(txt)
    (cfg["out"]/"verification_report.json").write_text(json.dumps(dict(
        config=cfg["name"], runtime_min=(time.time() - T_START) / 60, module_sha256=MODULE_SHA, gate_sha256=sha256_file(GATE_PATH), gpu=GPU_INFO,
        checks=[dict(level=l, name=n, detail=d) for l, n, d in CHECKS], errors=ERRORS, events=EVENTS, artifacts=dict(arts),
        verdict_sha256=hashlib.sha256((cfg["out"]/"gate_verdict.json").read_bytes()).hexdigest()), indent=2))
    return txt

def run_pipeline(cfg):
    global SEL_ROWS, TUNE_LOG
    print(f"===== run_pipeline({cfg['name']}) =====")
    o = cfg["out"]
    if cfg["name"] == "smoke":
        shutil.rmtree(o, ignore_errors=True); shutil.rmtree(cfg["cache"], ignore_errors=True)
    for p in (o, cfg["fig"], cfg["cache"]): p.mkdir(parents=True, exist_ok=True)
    LOGFILE[0] = o/"run_log.txt"; SEL_ROWS, TUNE_LOG = [], []
    gate_check(f"start {cfg['name']}"); ram_report(cfg["name"])
    sets = build_sets(cfg); print_estimates(cfg, sets)
    frames = []
    for st, wins, models in stage_plan(cfg):
        for win in wins:
            use_window(win)
            for model in models:
                frames.append(run_model(cfg, win, model, sets)); print(f"done {st}/{win}/{model}")
    P = pd.concat(frames, ignore_index=True)
    pd.concat(TUNE_LOG).to_csv(o/"tuning_log_5b.csv", index=False); pd.DataFrame(SEL_ROWS).to_csv(o/"selected_configs.csv", index=False)
    S = score_all(cfg, P); ST = stats(cfg, S); V = write_verdict(cfg, S, ST); figures(cfg, S, ST)
    gate_check(f"end {cfg['name']}")
    txt = build_report(cfg, S, ST, V)
    return dict(S=S, ST=ST, V=V, report_text=txt)

## Run: smoke first, then full

In [9]:
res_smoke = run_pipeline(SMOKE_CFG)
print("SMOKE finished:", {w: e.get("PASS") for w, e in res_smoke["V"]["windows"].items()}, "| errors:", ERRORS)
need(not ERRORS, "smoke completed without errors")
res_full = run_pipeline(FULL_CFG)
keep_awake(False)
print(res_full["report_text"])

===== run_pipeline(smoke) =====
[PASS] gate hash start smoke 
RAM available 5.0 GB smoke; top 8: python.exe 0.8GB, Code.exe 0.7GB, Code.exe 0.5GB, explorer.exe 0.4GB, MemCompression 0.4GB, claude.exe 0.4GB, Code.exe 0.4GB, MsMpEng.exe 0.3GB
[PASS] train/test disjoint (0,0) 
[PASS] no b3/sensitivity (0,0) 
[PASS] inner train/val disjoint (0,0,0) 
[PASS] tuning members exclude outer-test (0,0,0) 
[PASS] inner train/val disjoint (0,0,1) 
[PASS] tuning members exclude outer-test (0,0,1) 
[PASS] inner train/val disjoint (0,0,2) 
[PASS] tuning members exclude outer-test (0,0,2) 
[PASS] inner train/val disjoint (0,0,3) 
[PASS] tuning members exclude outer-test (0,0,3) 
[PASS] 4 inner folds (0,0) 
Estimated work (rate from smoke when available; rough):
  Stage A: units W300/M0:12, W300/M1p:12, W300/B3r:5, W300/B4r:5
[PASS] delta-Q check W300 row 36631 
[PASS] delta-Q check W300 row 4994 
[PASS] delta-Q check W300 row 43312 
[PASS] sub_idx gives 32 grid points 
RAM available 4.9 GB (W300); top 